# 03 · Inverse design — profile parameters from published coordinates

Port of blade2d's `tools/profile_inverse_fit.py` to C++: **side-aware residuals** (suction-side points are
measured against LE connector + SS + TE connector, pressure-side points against LE connector + PS + TE
connector), a **bounded Levenberg–Marquardt** solver (Eigen) in stages, and an optional **fairness penalty**
on concave dips of the edge-connector curvature, so a fit cannot hide an S-shaped edge.

Target: the published NACA 65-(12)10 section (blade2d `naca65/naca65_1210_XY_dense241.csv`, built from
NACA Table 6 with C_Li = 1.2, t/c = 0.10). blade2d reached RMS 0.0095 % chord on it.

In [ ]:
%matplotlib inline
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# the bladegen package lives in <build>/python (cmake --preset python); BLADEGEN_PY overrides
REPO = Path.cwd().resolve().parents[1]
for cand in [os.environ.get("BLADEGEN_PY", ""), REPO / "build" / "python" / "python"]:
    if cand and Path(cand).is_dir() and str(cand) not in sys.path:
        sys.path.insert(0, str(cand))
import bladegen as bg
import bladegen.notebook as nb
OUT = Path(os.environ.get("BLADEGEN_TEST_OUTPUT", REPO / "build" / "notebook_output"))
OUT.mkdir(parents=True, exist_ok=True)
print("BladeGen", bg.__version__, "| widgets:", nb.HAVE_WIDGETS)

In [ ]:
ss, ps = nb.read_sides_csv(str(REPO / "tests" / "data" / "naca65_1210_XY_dense241.csv"))
start = bg.ProfileParams.cda_start()            # a generic start, far from the answer
results = {}
for label, mode, w in [("blade2d connector, no fairness", "G2Legacy", 0.0),
                       ("Fair connector, no fairness", "Fair", 0.0),
                       ("Fair connector + fairness penalty", "Fair", 1.0)]:
    s0 = start.copy(); s0.connector = getattr(bg.ConnectorMode, mode)
    if mode == "G2Legacy":
        s0.le.tension, s0.te.tension = 0.28, 0.20
    opt = bg.FitOptions(); opt.params = nb.naca_search_box(s0); opt.fairness_weight = w
    t0 = time.perf_counter(); r = bg.fit_profile(ss, ps, s0, opt); dt = time.perf_counter() - t0
    q = bg.check_quality(bg.Profile2D(r["params"]))
    results[label] = (r, q)
    print(f"{label:<36} RMS {100 * r['rms']:.4f} % chord, max {100 * r['max']:.4f} %, "
          f"{r['evaluations']} evaluations, {dt:.1f} s, {q['errors']} errors / {q['warnings']} warnings")

The C++ fit needs a few seconds where the Python fit needed minutes. Below: data against the fitted
profile, the LE / TE zooms and the residual of every published point.

In [ ]:
r, q = results["Fair connector + fairness penalty"]
prof = bg.Profile2D(r["params"])
fig = plt.figure(figsize=(15, 9)); gs = fig.add_gridspec(2, 3, height_ratios=[1, 1])
ax = fig.add_subplot(gs[0, :]); nb.draw_profile(ax, prof, q, control=False)
ax.plot(*ss.T, "o", mfc="none", mec="k", ms=3.5, label="published"); ax.plot(*ps.T, "o", mfc="none", mec="k", ms=3.5)
nb._badge(ax, q); ax.legend(fontsize=8, loc="lower right")
for k, (c, lo, hi) in enumerate([("LE", -0.01, 0.05), ("TE", 0.95, 1.01)]):
    a = fig.add_subplot(gs[1, k]); nb.draw_profile(a, prof, q, control=False)
    a.plot(*ss.T, "o", mfc="none", mec="k", ms=4); a.plot(*ps.T, "o", mfc="none", mec="k", ms=4)
    a.set_xlim(lo, hi); a.set_ylim(-0.03, 0.04) if c == "LE" else a.set_ylim(-0.02, 0.02); a.set_title(c + " zoom")
a = fig.add_subplot(gs[1, 2])
a.vlines(np.arange(len(r["d_ss"])), 0, 100 * r["d_ss"], color=nb.C_SS, label="suction side")
a.vlines(len(r["d_ss"]) + 2 + np.arange(len(r["d_ps"])), 0, 100 * r["d_ps"], color=nb.C_PS, label="pressure side")
a.set_ylabel("distance [% chord]"); a.set_title(f"residuals (RMS {100 * r['rms']:.4f} %)"); a.legend(fontsize=8)
plt.show()

## Curvature quality of the three fits

Same accuracy, different geometry quality: without the penalty the fit reproduces the cusped NACA TE with an
S-shaped TE connector (error); with it the TE stays convex.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, (label, (r, q)) in zip(ax, results.items()):
    nb.draw_curvature(a, q, bg.Profile2D(r["params"]))
    a.set_title(f"{label}\n{q['errors']} errors, {q['warnings']} warnings", fontsize=9,
                color="red" if q["errors"] else "green")
plt.show()
bg.write_profile_json(results["Fair connector + fairness penalty"][0]["params"], str(OUT / "naca65_1210_fit.json"))